In [1]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')

In [3]:
# Load env variables and create client
import base64
from anthropic import Anthropic

client = Anthropic(api_key=my_anthropic_key,)
model = my_anthropic_model

In [6]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [4]:
# TODO: Read image data, feed into Claude

def read_image(image_file_name):
  #with open("image.png", "rb") as f:
  #  image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
  with open(f"images/{image_file_name}", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

  return image_bytes
#


In [7]:
messages = []

image_bytes = read_image("prop7.png")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)


Message(id='msg_011Cf4pygGa4c6VnjKNhBhU4', container=None, content=[TextBlock(citations=None, text="# Satellite Image Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a light-colored, rectangular structure with a gray roof located in the center of the property, accessible via a driveway from the left side, and clearly distinguishable as the main dwelling among smaller outbuildings on the property.\n\n**2. Tree Overhang Analysis:**\nApproximately 60-75% of the residence's roof is directly covered or overshadowed by dense tree canopy, with particularly heavy overhang concentrated over the central and eastern portions of the structure, creating significant branches directly above the roofline.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees present substantial wildfire vulnerability through multiple ember catch points where branches interface with the roof, potential bridges connecting the dense surrounding forest canopy directly to roof surfaces, an